# পাঠ ১৩ - এজেন্ট মেমরি


## সেটআপ

এই নোটবুকটি প্রদর্শন করে কীভাবে **Microsoft Agent Framework** (MAF) ব্যবহার করে **স্থিতিশীল মেমরি** সহ একটি ভ্রমণ বুকিং এজেন্ট তৈরি করা যায়।

আপনি শিখবেন কীভাবে বিভিন্ন ধরনের এজেন্ট মেমরি — ওয়ার্কিং, সংক্ষিপ্তমেয়াদি, এবং দীর্ঘমেয়াদি — এজেন্টকে তথ্য ধরে রাখার এবং কথোপকথনের মাধ্যমে তথ্য ব্যবহার করার প্রভাব ফেলে।

**প্রয়োজনীয়তা:**
- একটি Azure AI Foundry প্রকল্প যার সাথে একটি চ্যাট মডেল ডিপ্লয় করা হয়েছে (যেমন `gpt-4o-mini`)।
- Azure CLI দিয়ে লগইন করা — আপনার টার্মিনালে `az login` চালান।
- `AZURE_AI_PROJECT_ENDPOINT` — আপনার Azure AI Foundry প্রকল্পের এন্ডপয়েন্ট।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয় করা মডেলের নাম।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## এজেন্ট মেমোরির প্রকারভেদ

এআই এজেন্ট বিভিন্ন ধরনের মেমোরি ব্যবহার করতে পারে, যাদের প্রতিটি একটি ভিন্ন উদ্দেশ্য পূরণ করে:

### ওয়ার্কিং মেমোরি
কথোপকথনের থ্রেড নিজেই — একটি সেশনের মধ্যে বিনিময় হওয়া বার্তাসমূহ। এজেন্ট একই থ্রেডের পূর্বের বার্তাগুলোর দিকে ফিরে দেখতে পারে সামঞ্জস্য বজায় রাখতে। MAF-এ এটি **`agent.create_session()`** মাধ্যমে সৃষ্টি হয়, যা একটি `AgentSession` ফেরত দেয়।

### শর্ট-টার্ম মেমোরি
যে তথ্যটি একটি কাজ বা সেশনের সময়কাল ধরে থাকে কিন্তু স্থায়ীভাবে সংরক্ষিত হয় না। উদাহরণস্বরূপ, এজেন্ট একটি মাল্টি-টার্ন পরিকল্পনা কথোপকথনের সময় কিছু তথ্য সংগ্রহ করতে পারে এবং সেগুলো ব্যবহার করে একটি চূড়ান্ত ক্রমসূত্র প্রদান করতে পারে।

### লং-টার্ম মেমোরি
পছন্দসমূহ এবং তথ্য যা **সেশনগুলোর মধ্যে** টিকে থাকে। একজন ফিরে আসা ব্যবহারকারীকেও তার খাদ্যাভ্যাসের সীমাবদ্ধতা বা ভ্রমণ শৈলী পুনরাবৃত্তি করতে হবে না। লং-টার্ম মেমোরি সাধারণত একটি বাহ্যিক স্টোর দ্বারা সমর্থিত থাকে — একটি ডাটাবেস, ফাইল, বা ভেক্টর ইনডেক্স — এবং টুলের মাধ্যমে এজেন্টের কাছে উপস্থাপিত হয়।


## সেশন সহ ওয়ার্কিং মেমরি

মেমরির সবচেয়ে সহজ রূপ হল কথোপকথনের সেশন। যখন আপনি একই সেশন (যা `agent.create_session()` এর মাধ্যমে তৈরি করা হয়েছে) ধারাবাহিক `agent.run()` কলগুলিতে পাঠান, তখন এজেন্ট ওই কথোপকথনের পূর্ণ ইতিহাস দেখে এবং পূর্বের বিবরণ মনে রাখতে পারে।

চলুন একটি ট্রাভেল এজেন্ট তৈরি করি এবং ওয়ার্কিং মেমরি প্রদর্শন করি।


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

এজেন্টটি সঠিকভাবে বাজেটটির কথা মনে রেখেছিল কারণ দুটি বার্তারই একই সেশন শেয়ার করা হয়েছে। এটি হলো **কর্মস্মৃতি** — এটি শুধুমাত্র সেশনের জীবনকাল পর্যন্ত বিদ্যমান থাকে।

### নতুন থ্রেডে কী ঘটে?

যদি আমরা একটি **নতুন** সেশন তৈরি করি, তাহলে এজেন্টের পূর্ববর্তী কথোপকথনের কোনো স্মৃতি থাকবে না:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## দীর্ঘমেয়াদি স্মৃতি প্যাটার্ন

ব্যবহারকারীর পছন্দগুলি **সেশন জুড়ে** মনে রাখতে, আমাদের একটি টেকসই সঞ্চয়স্থল প্রয়োজন যা কথোপকথন থ্রেডের বাইরে থাকে। এজেন্ট এই সঞ্চয়স্থলে প্রবেশ করে **টুলস** এর মাধ্যমে — ফাংশনসমূহ যা এটি তথ্য সংরক্ষণ এবং পুনরুদ্ধার করার জন্য কল করতে পারে।

নীচে আমরা একটি সরল ইন-মেমরি পছন্দ সঞ্চয়স্থল বাস্তবায়ন করেছি (প্রোডাকশনে আপনি এটিকে একটি ডাটাবেস বা ভেক্টর ইনডেক্স দিয়ে ব্যাক করবেন) এবং এটিকে এমন টুলস হিসেবে প্রকাশ করেছি যা এজেন্ট ব্যবহার করতে পারে।

### আর্কিটেকচার
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — প্রথমবারের ব্যবহারকারী বার্ষিকীর সফর বুক করছেন

সারা প্রথমবার এখানে এসেছেন। এজেন্টকে টুলগুলোর মাধ্যমে তার পছন্দ সংরক্ষণ করতে হবে এবং সেগুলো ব্যবহার করে হোটেল প্রস্তাব দিতে হবে।


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### পরিস্থিতি ২ — সারা কয়েক সপ্তাহ পরে ফিরে আসে

সারা একটি **সম্পূর্ণ নতুন থ্রেড** শুরু করে (নতুন সেশনটি অনুকরণ করে)। ওয়ার্কিং মেমোরি খালি, কিন্তু দীর্ঘমেয়াদী পছন্দের দোকানে এখনও তার তথ্য আছে। এজেন্টকে এটি পুনরুদ্ধার করে ব্যক্তিগতকৃত পরামর্শ দেওয়ার জন্য ব্যবহার করা উচিত।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## সারসংক্ষেপ

এই পাঠে আপনি তিন ধরনের এজেন্ট মেমরি এবং কীভাবে মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক ব্যবহার করে সেগুলো বাস্তবায়ন করতে হয় তা শিখেছেন:

| মেমরি টাইপ | MAF পদ্ধতি | সময়কাল |
|---|---|---|
| **ওয়ার্কিং** | `agent.create_session()` | একক আলোচনা |
| **শর্ট-টার্ম** | একটি থ্রেডের মধ্যে সঞ্চিত প্রসঙ্গ | একক কাজ / সেশন |
| **লং-টার্ম** | বাহ্যিক স্টোর যা `@tool` ফাংশন দিয়ে অ্যাক্সেস করা হয় | বিভিন্ন সেশন জুড়ে |

### মূল বিষয়সমূহ
1. **`agent.create_session()`** ওয়ার্কিং মেমরি সরবরাহ করে — এজেন্ট সেশনের মধ্যে সম্পূর্ণ আলোচনা ইতিহাস দেখতে পায়।
2. **নতুন সেশনগুলো প্রসঙ্গ হারায়** — লং-টার্ম মেমরি ছাড়া এজেন্ট পূর্ববর্তী আলোচনা মনে রাখতে পারে না।
3. **`@tool` ফাংশনগুলো ফাঁক পূরণ করে** — এগুলো এজেন্টকে একটি টেকসই স্টোর থেকে তথ্য সংরক্ষণ ও পুনরুদ্ধার করতে দেয়।
4. **ব্যক্তিগতকরণ সময়ের সাথে উন্নত হয়** — যত বেশি পছন্দ সংরক্ষণ করা হয়, এজেন্টের পরামর্শ ততই উন্নত হয়।

### বাস্তব-জগতে প্রয়োগসমূহ
- **কাস্টমার সার্ভিস**: গ্রাহকের ইতিহাস এবং পছন্দ মনে রাখা
- **পার্সোনাল অ্যাসিস্ট্যান্ট**: দিন বা সপ্তাহ জুড়ে প্রসঙ্গ বজায় রাখা
- **হেলথকেয়ার**: রোগীর তথ্য এবং পছন্দ ট্র্যাক করা
- **ই-কমার্স**: ইতিহাসের ভিত্তিতে ব্যক্তিগতকৃত শপিং

### পরবর্তী ধাপসমূহ
- ইন-মেমরি ডিকশনরি পরিবর্তে একটি ডাটাবেস বা ভেক্টর স্টোর ব্যবহার করুন (যেমন Azure AI Search)
- সময়-সংবেদনশীল তথ্যের জন্য মেমরি মেয়াদ অন্তর্ভুক্ত করুন
- শেয়ার করা মেমরি সহ একাধিক এজেন্ট সিস্টেম তৈরি করুন
- জ্ঞান-গ্রাফ সমর্থিত মেমরির জন্য Cognee নোটবুকটি অন্বেষণ করুন


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**প্রত্যাহার**:
এই নথি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা ত্রুটি থাকতে পারে। মূল নথি তার নিজস্ব ভাষায়ই কর্তৃপক্ষের উৎস হিসেবে বিবেচিত হবে। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদ ব্যবহারে সৃষ্ট কোনো ভুলবোঝাপড়া বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
